# 02 — Devanagari Numeral Training (Transfer Learning)

This notebook fine-tunes the MNIST CNN from `01_MNIST_training.ipynb` to recognise
**Devanagari numerals (०–९)**, using the folder-per-digit dataset in
`../dataset/devanagari/` (`digit_0` … `digit_9`).

**Approach: transfer learning**
1. Load the pretrained `mnist_cnn.keras` model.
2. Freeze the convolutional / batch-norm layers (they already know general edge and
   stroke features).
3. Re-train only the dense head, with a low learning rate, on the Devanagari digit
   images.
4. Evaluate and save the result as `../models/devanagari_cnn.keras`.

The same OpenCV preprocessing pipeline used in `src/predict.py` is reused here so the
training data is processed identically to how live predictions will be.

## 1. Imports & configuration

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import keras
from keras import layers, regularizers
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Paths (relative to the notebooks/ folder)
DATASET_DIR = Path("../dataset/devanagari")   # expects digit_0 ... digit_9 subfolders
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRETRAINED_MODEL_PATH = MODELS_DIR / "mnist_cnn.keras"
SAVE_MODEL_PATH = MODELS_DIR / "devanagari_cnn.keras"

print("Keras version:", keras.__version__)


## 2. Preprocessing pipeline

Identical to `src/predict.py` / `src/collect_data.py`, so training and inference see the
same kind of input:

1. Grayscale + Gaussian blur (denoise)
2. Otsu thresholding (binarise, invert so digit is white on black)
3. Crop to the digit's bounding box (with a small pad)
4. Resize to fit inside 20×20 while preserving aspect ratio
5. Paste onto a centred 28×28 black canvas
6. Normalise to `[0, 1]`

In [ ]:
def preprocess_image(image_path):
    """Grayscale -> denoise -> threshold -> crop -> resize -> center on 28x28."""
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    img = cv2.GaussianBlur(img, (5, 5), 0)
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    coords = cv2.findNonZero(img)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = 4
        img = img[max(0, y - pad):min(img.shape[0], y + h + pad),
                  max(0, x - pad):min(img.shape[1], x + w + pad)]

    if img.size == 0:
        return None

    h, w = img.shape
    aspect = w / h
    if aspect > 1:
        new_w, new_h = 20, max(1, int(20 / aspect))
    else:
        new_h, new_w = 20, max(1, int(20 * aspect))

    img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    final_img = np.zeros((28, 28), dtype=np.uint8)
    final_img[(28 - new_h) // 2: (28 - new_h) // 2 + new_h,
              (28 - new_w) // 2: (28 - new_w) // 2 + new_w] = img

    return final_img.astype("float32") / 255.0


## 3. Load the Devanagari dataset

Walks `digit_0` … `digit_9` subfolders, preprocessing every image.

In [ ]:
def load_devanagari_dataset(base_path):
    X, y = [], []
    for digit in range(10):
        digit_folder = Path(base_path) / f"digit_{digit}"
        if not digit_folder.exists():
            print(f"Warning: {digit_folder} not found, skipping.")
            continue
        count_before = len(X)
        for img_path in digit_folder.glob("*.*"):
            processed = preprocess_image(img_path)
            if processed is not None:
                X.append(processed)
                y.append(digit)
        print(f"digit_{digit}: {len(X) - count_before} images")

    X = np.expand_dims(np.array(X), -1)
    y = np.array(y)
    return X, y

X_dev, y_dev = load_devanagari_dataset(DATASET_DIR)
print(f"\nLoaded {len(X_dev)} images total, shape={X_dev.shape}")


### Peek at a few preprocessed samples

In [ ]:
if len(X_dev) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    idx = np.random.choice(len(X_dev), size=min(10, len(X_dev)), replace=False)
    for ax, i in zip(axes.flat, idx):
        ax.imshow(X_dev[i].squeeze(), cmap="gray")
        ax.set_title(str(y_dev[i]))
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No images loaded — check DATASET_DIR path.")


## 4. Train / validation split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, random_state=42, stratify=y_dev
)
print("Train:", X_train.shape, " Val:", X_val.shape)


## 5. Load the pretrained MNIST model and freeze the conv backbone

Only the `Conv2D` and `BatchNormalization` layers are frozen — the dense head is
re-trained from its MNIST-tuned starting point on the Devanagari digits.

In [ ]:
model = keras.models.load_model(PRETRAINED_MODEL_PATH)

for layer in model.layers:
    if isinstance(layer, (layers.Conv2D, layers.BatchNormalization)):
        layer.trainable = False

model.summary()


## 6. Re-compile with a lower learning rate

Fine-tuning needs a gentler learning rate than training from scratch.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)


## 7. Callbacks

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
]


## 8. Fine-tune

In [ ]:
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
)


## 9. Evaluate

In [ ]:
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss:     {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")


## 10. Accuracy / loss curves

Appended/overwritten alongside the MNIST curves in `../results/`.

In [ ]:
hist = history.history

plt.figure(figsize=(6, 4))
plt.plot(hist["accuracy"], label="train")
plt.plot(hist["val_accuracy"], label="val")
plt.title("Devanagari — Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "devanagari_accuracy_graph.png", dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(hist["loss"], label="train")
plt.plot(hist["val_loss"], label="val")
plt.title("Devanagari — Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "devanagari_loss_graph.png", dpi=150)
plt.show()


## 11. Confusion matrix & classification report

In [ ]:
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=range(10), yticklabels=range(10))
plt.title("Devanagari — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "devanagari_confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(y_val, y_pred, digits=4))


## 12. Save the fine-tuned model

In [ ]:
model.save(SAVE_MODEL_PATH)
print(f"Devanagari model saved to {SAVE_MODEL_PATH}")
